<a href="https://colab.research.google.com/github/arinamartynyuk/-4-9/blob/main/%D0%9B%D0%A05_%D1%87%D0%B0%D1%81%D0%BE%D0%B2%D1%96_%D1%80%D1%8F%D0%B4%D0%B8_%D0%B3%D0%BE%D1%82%D0%BE%D0%B2%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Daily Weather Time Series Data - Delhi, India**

https://www.kaggle.com/datasets/yug201/daily-climate-time-series-data-delhi-india?select=kaggel_weather_2013_to_2024.csv

**Тема:** Прогнозування часових рядів на основі кліматичних даних

**Мета роботи:**
Навчитися виконувати аналіз часових рядів, проводити дослідницький аналіз даних (EDA), визначати автокореляційні залежності, будувати моделі прогнозування та оцінювати їхню точність.

**Вихідні дані**

Для виконання роботи використовується датасет Daily Climate Time Series Data (Delhi, India) з платформи Kaggle, який містить щоденні кліматичні показники (температура, вологість, опади тощо) за кілька років.


1. Завантажити датасет з Kaggle.

2. Імпортувати дані у середовище Python (Jupyter Notebook або Google Colab).

3. Ознайомитися зі структурою датасету (head(), info(), describe()).

4. Попередня обробка даних

-Перетворити колонку з датою у формат datetime.

-Відсортувати дані за датою.

-Встановити дату як індекс часового ряду.

-Перевірити наявність пропущених значень та за необхідності обробити їх.

5. Дослідницький аналіз даних (EDA)

-Побудувати графік зміни температури у часі.

-Побудувати ковзне середнє значення температури.

-Побудувати гістограму розподілу температури.

-Побудувати boxplot для виявлення можливих викидів.

-Аналіз автокореляції

Побудувати графік автокореляційної функції (ACF).

Побудувати графік часткової автокореляції (PACF).

-Проаналізувати залежність значень часового ряду від попередніх періодів.

6.Підготовка даних для моделювання

-Розділити дані на тренувальну та тестову вибірки (наприклад, 80% / 20%).

7. Побудувати щонайменше три моделі прогнозування:

-ARIMA (класична статистична модель);

-Prophet (модель прогнозування часових рядів від Meta);

-LSTM (нейронна мережа для послідовних даних).

8. Оцінка моделей
Для кожної моделі обчислити метрики:

-MAE (Mean Absolute Error)

-RMSE (Root Mean Squared Error)

-MAPE (Mean Absolute Percentage Error)

8.Візуалізація результатів

-Побудувати графіки прогнозованих та фактичних значень.

-Побудувати порівняльний графік прогнозів усіх моделей.

-Порівняння моделей

-Створити таблицю з метриками точності.

9.Визначити модель, яка показала найкращий результат.

In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True


In [ ]:
import os
from pathlib import Path

csv_candidates = list(Path('/content').glob('*.csv')) if os.path.exists('/content') else []
if csv_candidates:
    csv_path = str(csv_candidates[0])
else:
    try:
        import requests
        url = 'https://www.kaggle.com/api/v1/datasets/download/yug201/daily-climate-time-series-data-delhi-india'
        r = requests.get(url, timeout=30)
        if r.ok and ('zip' in r.headers.get('content-type','').lower() or r.content[:2] == b'PK'):
            import zipfile, io
            z = zipfile.ZipFile(io.BytesIO(r.content))
            csv_files = [n for n in z.namelist() if n.lower().endswith('.csv')]
            if csv_files:
                z.extract(csv_files[0], '/content')
                csv_path = '/content/' + csv_files[0]
            else:
                raise Exception('У ZIP не знайдено CSV')
        else:
            raise Exception('Не вдалося завантажити CSV автоматично')
    except Exception:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next(iter(uploaded.keys()))

print('Файл:', csv_path)


In [ ]:
# Зчитування даних та первинний огляд
df = pd.read_csv(csv_path)
display(df.head())
print('Розмір датасету:', df.shape)
df.info()
display(df.describe(include='all'))


Підготовка даних

In [ ]:
# Попередня обробка: дата, сортування, індекс
# Знаходимо колонку з датою за назвою
date_candidates = [c for c in df.columns if c.lower() in ['date', 'datetime', 'time']]
date_col = date_candidates[0] if date_candidates else df.columns[0]
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df = df.dropna(subset=[date_col]).sort_values(date_col)

# Видаляємо повністю порожні та дубльовані рядки
df = df.dropna(how='all').drop_duplicates()

# Знаходимо колонку температури
temp_candidates = [c for c in df.columns if any(k in c.lower() for k in ['temp', 'temperature'])]
if not temp_candidates:
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_cols:
        raise ValueError('У датасеті не знайдено числових колонок для аналізу.')
    temp_col = numeric_cols[0]
else:
    temp_col = temp_candidates[0]

df = df.set_index(date_col)
df = df[~df.index.duplicated(keep='first')]
df[temp_col] = pd.to_numeric(df[temp_col], errors='coerce')
df[temp_col] = df[temp_col].interpolate(method='time').ffill().bfill()
print('Колонка дати:', date_col)
print('Цільова колонка:', temp_col)
display(df.head())


In [ ]:
# Інформація про структуру датасету
df.info()
display(df.describe())


In [ ]:
# Перевірка пропущених значень
display(df.isna().sum())


Графік температури

In [ ]:
# Графік зміни температури у часі
plt.figure()
plt.plot(df.index, df[temp_col], label='Температура')
plt.title('Зміна температури у часі')
plt.xlabel('Дата')
plt.ylabel(temp_col)
plt.legend()
plt.show()


Ковзне середнє

In [ ]:
# Ковзне середнє (30 днів)
rolling_mean = df[temp_col].rolling(window=30, min_periods=1).mean()
plt.figure()
plt.plot(df.index, df[temp_col], alpha=0.5, label='Фактичні дані')
plt.plot(df.index, rolling_mean, linewidth=2, label='Ковзне середнє (30)')
plt.title('Температура та ковзне середнє')
plt.xlabel('Дата')
plt.ylabel(temp_col)
plt.legend()
plt.show()


Розподіл температури

In [ ]:
# Гістограма розподілу температури
plt.figure()
plt.hist(df[temp_col], bins=30, edgecolor='black')
plt.title('Розподіл температури')
plt.xlabel(temp_col)
plt.ylabel('Частота')
plt.show()


In [ ]:
# Boxplot для виявлення можливих викидів
plt.figure()
sns.boxplot(x=df[temp_col])
plt.title('Boxplot температури')
plt.xlabel(temp_col)
plt.show()


Аналіз автокореляції
ACF

PACF

In [ ]:
# ACF: автокореляційна функція
series = df[temp_col].dropna()
lags = min(40, max(1, len(series)//2 - 1))
plot_acf(series, lags=lags)
plt.title('Автокореляційна функція (ACF)')
plt.show()


Автокореляційна функція (ACF) використовується для аналізу залежності значень часового ряду від його попередніх значень. Вона відображає ступінь кореляції між значенням показника у певний момент часу та значеннями цього самого показника у попередні моменти часу, які називаються лагами. Побудова графіка автокореляційної функції дозволяє виявити, чи існує статистично значуща залежність між поточним значенням ряду та його попередніми спостереженнями. На такому графіку по горизонтальній осі відкладаються лаги, тобто кількість часових кроків назад, а по вертикальній — значення коефіцієнта автокореляції. Аналіз графіка ACF дає можливість виявити наявність тренду, сезонності або інших закономірностей у часовому ряді, а також використовується для визначення параметрів статистичних моделей прогнозування, зокрема моделі ARIMA.

In [ ]:
# PACF: часткова автокореляційна функція
plot_pacf(series, lags=min(lags, max(1, len(series)//2 - 1)), method='ywm')
plt.title('Часткова автокореляційна функція (PACF)')
plt.show()


Часткова автокореляційна функція (PACF) застосовується для більш точного визначення залежності між значеннями часового ряду. На відміну від звичайної автокореляційної функції, PACF вимірює кореляцію між значенням ряду у певний момент часу та значенням у попередній момент часу з конкретним лагом, усуваючи вплив усіх проміжних лагів. Таким чином, часткова автокореляція показує «чисту» залежність між двома спостереженнями часового ряду. Побудова графіка PACF дозволяє визначити, які саме лаги мають статистично значущий вплив на значення ряду. Це є важливим етапом під час побудови моделей прогнозування часових рядів, оскільки аналіз графіка часткової автокореляції використовується для визначення порядку авторегресійної складової моделі, зокрема параметра
𝑝
p у моделі ARIMA.

In [ ]:
# Формування цільового ряду та часовий train/test split (80/20)
y = df[temp_col].astype(float).dropna()
split_idx = int(len(y) * 0.8)
train = y.iloc[:split_idx]
test = y.iloc[split_idx:]
print(f'Train: {len(train)} значень, Test: {len(test)} значень')
print(f'Період train: {train.index.min()} — {train.index.max()}')
print(f'Період test: {test.index.min()} — {test.index.max()}')


ARIMA

https://developer.ibm.com/tutorials/awb-arima-models-in-python/

In [ ]:
# ARIMA: навчання та прогноз для тестової вибірки
arima_model = ARIMA(train, order=(5, 1, 2))
arima_fit = arima_model.fit()
arima_pred = arima_fit.forecast(steps=len(test))
arima_pred.index = test.index
display(arima_pred.head())


In [ ]:
# Статистичний звіт ARIMA
print(arima_fit.summary())


сигма 2 : це представляє дисперсію залишкових значень, чим менше, тим краще

логарифмічна правдоподібність : це оцінка максимальної правдоподібності для моделі ARIMA, чим вище, тим краще

aic : міра ступеню відповідності та економності моделі, чим нижче, тим краще

In [ ]:
# Аналіз залишків ARIMA та графіки
residuals = arima_fit.resid
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(residuals)
axes[0].set_title('Залишки ARIMA')
axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Розподіл залишків ARIMA')
plt.tight_layout()
plt.show()

display(acorr_ljungbox(residuals, lags=[10], return_df=True))


In [ ]:
# Метрики ARIMA
def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    denom = np.where(np.abs(actual) < 1e-8, np.nan, np.abs(actual))
    mape = np.nanmean(np.abs((actual-predicted)/denom))*100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

arima_metrics = calculate_metrics(test, arima_pred)
print(arima_metrics)


In [ ]:
# Візуалізація фактичних значень та прогнозу ARIMA
plt.figure()
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Фактичні значення')
plt.plot(test.index, arima_pred, label='ARIMA прогноз')
plt.title('ARIMA: фактичні значення та прогноз')
plt.xlabel('Дата')
plt.ylabel(temp_col)
plt.legend()
plt.show()


PROPHET

https://facebook.github.io/prophet/docs/quick_start.html

In [ ]:
# Підготовка даних для Prophet (ds, y)
prophet_train = pd.DataFrame({'ds': train.index, 'y': train.values})
prophet_test = pd.DataFrame({'ds': test.index, 'y': test.values})
display(prophet_train.head())


In [ ]:
# Створення та навчання Prophet
!pip -q install prophet
from prophet import Prophet

prophet_model = Prophet()
prophet_model.fit(prophet_train)


Прогноз

In [ ]:
# Прогноз Prophet на тестовий період
prophet_forecast = prophet_model.predict(prophet_test[['ds']])
prophet_pred = pd.Series(prophet_forecast['yhat'].values, index=test.index)
display(prophet_forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head())


In [ ]:
# Метрики Prophet
prophet_metrics = calculate_metrics(test, prophet_pred)
print(prophet_metrics)


In [ ]:
# Фактичні значення та прогноз Prophet
plt.figure()
plt.plot(test.index, test, label='Фактичні значення')
plt.plot(test.index, prophet_pred, label='Prophet прогноз')
plt.title('Prophet: фактичні значення та прогноз')
plt.xlabel('Дата')
plt.ylabel(temp_col)
plt.legend()
plt.show()


In [ ]:
# Компоненти моделі Prophet
prophet_model.plot_components(prophet_forecast)
plt.show()


ПІДГОТОВКА ДАНИХ ДЛЯ LSTM

In [ ]:
# Масштабування та формування послідовностей для LSTM
!pip -q install tensorflow
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train.values.reshape(-1, 1))
# Для тесту використовуємо історичний хвіст train, щоб створити перші послідовності
all_values = y.values.reshape(-1, 1)
all_scaled = scaler.transform(all_values)
sequence_length = min(30, max(5, len(train)//10))
X_all, y_all = [], []
for i in range(sequence_length, len(all_scaled)):
    X_all.append(all_scaled[i-sequence_length:i, 0])
    y_all.append(all_scaled[i, 0])
X_all = np.array(X_all).reshape(-1, sequence_length, 1)
y_all = np.array(y_all)
print('X shape:', X_all.shape, 'y shape:', y_all.shape)


In [ ]:
# Train/test для LSTM, без перемішування
# Перші послідовності, що відповідають train, та останні, що відповідають test
lstm_split = split_idx - sequence_length
X_train_lstm = X_all[:lstm_split]
y_train_lstm = y_all[:lstm_split]
X_test_lstm = X_all[lstm_split:]
y_test_lstm = y_all[lstm_split:]
print('Train sequences:', X_train_lstm.shape)
print('Test sequences:', X_test_lstm.shape)


In [ ]:
# Архітектура LSTM
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(sequence_length, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm_model.summary()


In [ ]:
# Навчання LSTM
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=30, batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    shuffle=False, verbose=1
)


In [ ]:
# Структура LSTM-моделі
lstm_model.summary()


In [ ]:
# Прогноз LSTM та зворотне масштабування
lstm_scaled_pred = lstm_model.predict(X_test_lstm, verbose=0)
lstm_pred_values = scaler.inverse_transform(lstm_scaled_pred).flatten()
lstm_actual_values = scaler.inverse_transform(y_test_lstm.reshape(-1,1)).flatten()


In [ ]:
# Часові мітки тестового прогнозу LSTM
lstm_test_index = test.index
lstm_pred = pd.Series(lstm_pred_values, index=lstm_test_index)
lstm_actual = pd.Series(lstm_actual_values, index=lstm_test_index)
print('Дати тестового прогнозу:', lstm_test_index[0], '—', lstm_test_index[-1])


In [ ]:
# Метрики LSTM
lstm_metrics = calculate_metrics(lstm_actual, lstm_pred)
print(lstm_metrics)


In [ ]:
# Фактичні значення та прогноз LSTM
plt.figure()
plt.plot(lstm_actual.index, lstm_actual, label='Фактичні значення')
plt.plot(lstm_pred.index, lstm_pred, label='LSTM прогноз')
plt.title('LSTM: фактичні значення та прогноз')
plt.xlabel('Дата')
plt.ylabel(temp_col)
plt.legend()
plt.show()


Порівняння моделей

In [ ]:
# Порівняльна таблиця метрик моделей
results = pd.DataFrame({
    'ARIMA': arima_metrics,
    'Prophet': prophet_metrics,
    'LSTM': lstm_metrics
}).T
display(results.round(4))

# Порівняння прогнозів на одному графіку
plt.figure(figsize=(14,6))
plt.plot(test.index, test, label='Фактичні значення', linewidth=2)
plt.plot(test.index, arima_pred, label='ARIMA')
plt.plot(test.index, prophet_pred, label='Prophet')
plt.plot(lstm_pred.index, lstm_pred, label='LSTM')
plt.title('Порівняння прогнозів моделей')
plt.xlabel('Дата')
plt.ylabel(temp_col)
plt.legend()
plt.show()

best_model = results['MAE'].idxmin()
print(f'Найкраща модель за MAE: {best_model}')


## Висновок

У лабораторній роботі було проведено аналіз кліматичного часового ряду: дані підготовлено та очищено, побудовано графіки зміни показника, ковзного середнього, гістограму й boxplot. За допомогою ACF та PACF досліджено автокореляційні залежності.

Для прогнозування було побудовано три моделі: ARIMA, Prophet та LSTM. Їх точність оцінено за MAE, RMSE та MAPE, а результати зіставлено у таблиці й на графіку. Найкращою вважається модель з найменшим значенням MAE (також враховуються RMSE і MAPE). Фактичний переможець визначається після виконання ноутбука за отриманими метриками.
